# f696 原账户的完美事后修补效果量下界

只读原生 Qlib 0bp Tiered 账户分量。固定持仓与费用，按预先冻结的三个组件集合计算完全事后、非交易性负贡献抹除比例；不评估风险或统计硬门。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
import hashlib,json
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R

root=Path.cwd();card=root/"research/stage1_f696_static_oracle_effect_size_20261024"
plan_path=card/"plan.json";plan=json.loads(plan_path.read_text());src=plan["source"]
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan["frozen"] and all(v==0 for v in plan["budget"].values())
source=root/src["card"]
assert sha(source/"summary.json")==src["summary_sha256"]
ss=json.loads((source/"summary.json").read_text())
assert ss["nas_evidence"]["sha256"]==src["nas_sha256"] and ss["nas_evidence"]["size_bytes"]==src["nas_size_bytes"]
assert ss["nas_readback"]["verified"] and ss["nas_readback"]["sha256"]==src["nas_sha256"]
artifacts=source/"qlib_recorder"/src["experiment_id"]/src["recorder_id"]/"artifacts"
for name,digest in src["artifact_sha256"].items():assert sha(artifacts/name)==digest,name
assert sha(root/src["provider"]/"qlib_store_manifest.json")==src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root/src["provider"]).resolve()),region="us",expression_cache=None,dataset_cache=None,kernels=1)
with R.uri_context(str(source/"qlib_recorder")):
 rec=R.get_recorder(experiment_id=src["experiment_id"],recorder_id=src["recorder_id"])
 daily=rec.load_object("analysis/daily").sort_values("trade_date").reset_index(drop=True)
 rolling=rec.load_object("analysis/rolling126")
 periods=rec.load_object("analysis/periods")
assert len(daily)==944 and len(rolling)==819 and len(periods.loc[periods.kind.eq("quarter")])==16
complete=periods.loc[periods.kind.eq("year") & ~periods.partial_boundary_period,"period"].astype(str).tolist()
partial=periods.loc[periods.kind.eq("year") & periods.partial_boundary_period,"period"].astype(str).tolist()
assert complete==["2024","2025"] and partial==["2023","2026"]
calendar=pd.DatetimeIndex(D.calendar(start_time="2023-01-03",end_time="2026-10-09",freq="day"))
assert pd.DatetimeIndex(daily.signal_date).is_unique and pd.DatetimeIndex(daily.signal_date).isin(calendar).all()
parts=["cash_opportunity_log","SOXL_selection_log","Stage1_direct_fees_log","timing_and_benchmark_residual_log"]
closure=float((daily[parts].sum(axis=1)-daily.paired_log_excess).abs().max())
assert closure<1e-12
cash=np.maximum(-daily.cash_opportunity_log.to_numpy(float),0)
selection=np.maximum(-daily.SOXL_selection_log.to_numpy(float),0)
base=daily.paired_log_excess.to_numpy(float)
static=pd.concat({"feature":pd.DataFrame({"cash_negative_erasable":cash,"selection_negative_erasable":selection}),
                  "label":pd.DataFrame({"paired_log_excess":base}),
                  "meta":pd.DataFrame({"trade_year":pd.to_datetime(daily.trade_date).dt.year.to_numpy()})},axis=1)
static.index=pd.MultiIndex.from_arrays([pd.DatetimeIndex(daily.signal_date),["SOXL_SOXX_CASH"]*len(daily)],names=["datetime","instrument"])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={"diagnostic":["2023-01-03","2026-10-07"]})
assert len(dataset.prepare("diagnostic",col_set=["feature","label","meta"]))==944
window_base=np.convolve(base,np.ones(126),mode="valid")
assert len(window_base)==819
max_window_error=0.0
for i,rw in enumerate(rolling.itertuples()):
 assert pd.Timestamp(daily.trade_date.iloc[i])==pd.Timestamp(rw.first_trade)
 max_window_error=max(max_window_error,abs(float(window_base[i])-float(rw.relative_log_excess)))
assert max_window_error<1e-10
assert int((window_base>0).sum())==255
trade_year=pd.to_datetime(daily.trade_date).dt.year.to_numpy()
def strict_break_even(original,capacity):
 if original>0:return 0.0
 if capacity<=0:return float("inf")
 return float(-original/capacity)
def route(name,capacity):
 full_base=float(base.sum());full_capacity=float(capacity.sum())
 wealth_threshold=strict_break_even(full_base,full_capacity)
 annual=[]
 for year in (2024,2025):
  mask=trade_year==year
  orig=float(base[mask].sum());cap=float(capacity[mask].sum())
  annual.append({"year":year,"base_log":orig,"erasable_negative_log":cap,"break_even_fraction_infimum":strict_break_even(orig,cap)})
 annual_threshold=max(x["break_even_fraction_infimum"] for x in annual)
 wc=np.convolve(capacity,np.ones(126),mode="valid")
 threshold=np.array([strict_break_even(float(l),float(c)) for l,c in zip(window_base,wc)])
 window_threshold=float(np.sort(threshold)[573])
 required=max(wealth_threshold,annual_threshold,window_threshold)
 full_one=base+capacity
 full_one_windows=window_base+wc
 one={"full_relative_wealth":float(np.exp(full_one.sum())),"complete_year_positive":int(sum(full_one[trade_year==year].sum()>0 for year in (2024,2025))),
      "positive_rolling126":int((full_one_windows>0).sum())}
 feasible=bool(np.isfinite(required) and required<=1)
 test=None
 if feasible:
  f=float(np.nextafter(required,np.inf))
  adjusted=base+f*capacity
  test={"fraction_just_above_infimum":f,"full_relative_wealth":float(np.exp(adjusted.sum())),
        "complete_year_positive":int(sum(adjusted[trade_year==year].sum()>0 for year in (2024,2025))),
        "positive_rolling126":int((window_base+f*wc>0).sum())}
 return {"route":name,"full_base_log":full_base,"erasable_negative_log":full_capacity,
         "wealth_break_even_fraction_infimum":wealth_threshold,"annual":annual,
         "annual_break_even_fraction_infimum":annual_threshold,"rolling574_break_even_fraction_infimum":window_threshold,
         "joint_reported_gate_fraction_infimum":required,"feasible_at_or_below_full_erasure":feasible,
         "full_erasure_static":one,"just_above_infimum_static":test}
routes=[route("cash_only",cash),route("selection_only",selection),route("cash_and_selection",cash+selection)]
result={"status":"read_only_static_oracle_effect_size_no_policy",
        "plan_sha256":sha(plan_path),"source_recorder_id":src["recorder_id"],
        "sessions":len(daily),"complete_years":complete,"partial_years":partial,"quarters":16,"rolling126_windows":len(rolling),
        "daily_component_closure_max_abs":closure,"source_rolling_sum_max_abs":max_window_error,
        "original_full_relative_wealth":float(np.exp(base.sum())),"original_positive_rolling126":255,
        "required_positive_rolling126":574,"routes":routes,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "policy_inference":False,"independent_validation":False,"SOXX_goal_achieved":False,
        "limits":["Outcome-known erasure of only negative daily components is a perfect hindsight oracle, not a decision-time feature",
                  "The old account path, holdings, own-state risk feedback, direct fees and NAV are held fixed; this is not a new native account",
                  "Available/full-minute drawdown, CVaR5, corrected block lower, 2/5bp and forward evidence are not evaluated",
                  "All 819 rolling windows overlap and cannot be treated as independent trials"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"routes":[{k:x[k] for k in ["route","joint_reported_gate_fraction_infimum","feasible_at_or_below_full_erasure","full_erasure_static"]} for x in routes]})


{'routes': [{'route': 'cash_only', 'joint_reported_gate_fraction_infimum': 0.22774891591120097, 'feasible_at_or_below_full_erasure': True, 'full_erasure_static': {'full_relative_wealth': 48.626006807890406, 'complete_year_positive': 2, 'positive_rolling126': 819}}, {'route': 'selection_only', 'joint_reported_gate_fraction_infimum': 0.3354156043797136, 'feasible_at_or_below_full_erasure': True, 'full_erasure_static': {'full_relative_wealth': 6.058746963831114, 'complete_year_positive': 2, 'positive_rolling126': 796}}, {'route': 'cash_and_selection', 'joint_reported_gate_fraction_infimum': 0.13437817932211732, 'feasible_at_or_below_full_erasure': True, 'full_erasure_static': {'full_relative_wealth': 429.46050540072315, 'complete_year_positive': 2, 'positive_rolling126': 819}}]}


## 中文结论

只读精确f696 A0、0bp额外滑点且扣Tiered费用的原生Qlib账户：944日、16季度、2024/2025两个完整年、2023/2026两个不完整年与819个连续126日窗口；日分量与原窗口对数超额闭合。固定原持仓、风险状态、费用与NAV路径，只让一个**事后知道当天分量正负**的假想者，等比例抹去负现金损失、负SOXL相对选择损失或两者。

在这项不可能实时获得的逐日上帝视角下，仅就SOXX财富>1、两个完整年均为正、至少574/819个窗口为正，所需均匀抹去比例的下确界分别是：仅现金 **22.77%**、仅SOXL选择 **33.54%**、两者合计 **13.44%**。紧高于该比例时三个静态会计门才达到，原账户为255/819窗、财富比0.686。这些比例**不是因子准确率、可实现的交易改进或晋升结果**；可用标记/全分钟回撤、CVaR5、校正区块下界、2/5bp及独立前向均未计算。

此前双现金层界限在每个126日窗口上抹去**整窗净负现金分量**，两层全抹也只到588窗；本卡改成逐日挑出每一个负日，甚至在整窗净现金为正时也能跳过有利保护日，因此满额逐日抹除可达819窗。两者不是冲突的策略结果，而是不同强度的事后选择权。不能用本卡的较小百分比声称“稍微改一下因子就够了”。

决定 `descriptive_per_day_hindsight_oracle_bound_not_a_factor_or_account`。下一项因子必须在决策时预测**何时**现金是保护、何时是拖累，还要在自己的完整账户里保住好区间与风险约束；不能把本事后下界当作参数目标。0拟合、0新账户、0独立验证，SOXX v2目标未达成。
